# Bin counting: encode a category by its target rate

_Feature Engineering (Zheng & Casari)_

**Replace a giant category (device ID, zip) with the click rate it has historically produced.**

Some categorical columns are enormous. In click prediction (the book's running example, on
       advertising data like the Avazu click-through set) a single column might be the device ID,
       the ad ID, or the app &mdash; with millions of distinct values. One-hot encoding
       turns that into millions of columns, almost all zero. That is wasteful and most models choke on it.

       Bin counting (also called target encoding or response-rate encoding) replaces
       the category's identity with statistics of the target for that category. Instead of a
       one-hot vector that says "this is device #8,412,067", you store one number: the historical
       click-through rate of that device, $P(\text{click}\mid \text{device})$. A million sparse
       columns collapse into a handful of dense, highly predictive ones.

---

This notebook is a practice scaffold for the **Bin counting: encode a category by its target rate** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — pandas + numpy + scikit-learn

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

# --- Load Avazu-style click data (book example: advertising click-through) ---
# Get it from Kaggle 'Avazu CTR' / the book repo:
#   github.com/alicezheng/feature-engineering-book
# Columns include high-cardinality IDs like 'device_id', 'app_id', 'site_id'
# and a binary target 'click' (1 = clicked, 0 = not).
df = pd.read_csv('avazu_train.csv', usecols=['device_id', 'click'])

col, target = 'device_id', 'click'
prior = df[target].mean()          # global click rate p0  (the back-off / fallback)
alpha = 20.0                       # additive-smoothing strength

# === Plain bin counting (counts -> rate -> log-odds) on the WHOLE column ===
# (Shown for clarity; for modeling use the out-of-fold version below.)
grp = df.groupby(col)[target]
counts = grp.agg(N1='sum', N=('count'))      # clicks and total per category value
counts['N0'] = counts['N'] - counts['N1']
# Conditional probability with Laplace smoothing -> shrinks rare values to the prior:
counts['rate'] = (counts['N1'] + alpha * prior) / (counts['N'] + alpha)
# Log-odds: a spread-out, model-friendly version of the rate.
counts['log_odds'] = np.log(counts['rate'] / (1.0 - counts['rate']))

# === Out-of-fold bin counting (the leakage-safe version you actually train on) ===
# Each row is encoded using ONLY the other folds -> no row sees its own label.
def out_of_fold_target_encode(df, col, target, alpha=20.0, n_splits=5, seed=0):
    oof = pd.Series(np.nan, index=df.index)
    prior = df[target].mean()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(df):
        tr, val = df.iloc[tr_idx], df.iloc[val_idx]
        g = tr.groupby(col)[target].agg(N1='sum', N='count')
        rate = (g['N1'] + alpha * prior) / (g['N'] + alpha)   # smoothed rate
        # map onto the held-out fold; UNSEEN values fall back to the global rate:
        oof.iloc[val_idx] = val[col].map(rate).fillna(prior).values
    return oof

df['device_ctr'] = out_of_fold_target_encode(df, col, target, alpha=alpha)
# 'device_ctr' is now ONE dense, highly predictive column replacing a million-wide one-hot.

# For brand-new IDs at SERVING time, encode with the full-train smoothed table and
# default to the prior:
serve_rate = counts['rate']
def encode_at_serving(device_id):
    return serve_rate.get(device_id, prior)   # unseen -> global click rate p0

## Visualize the data & results

_On a real dataset, does encoding a category by its OUT-OF-FOLD target rate beat a label-encoded baseline — and how dense and predictive is that single bin-counted column?_

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

d = load_breast_cancer()
ri = list(d.feature_names).index('mean radius')
x = d.data[:, ri]
y = d.target                                  # 1 = benign, 0 = malignant

# Make a 'high-cardinality category' by binning one feature into 5 buckets.
edges = np.percentile(x, [0, 20, 40, 60, 80, 100])
cat = np.clip(np.digitize(x, edges[1:-1]), 0, 4)   # category id in {0..4}

prior = y.mean()
alpha = 10.0

# --- Out-of-fold target-rate encoding (leakage-safe bin counting) ---
def oof_encode(cat, y, alpha, seed=0):
    enc = np.full(len(y), np.nan)
    skf = StratifiedKFold(5, shuffle=True, random_state=seed)
    for tr, va in skf.split(cat.reshape(-1, 1), y):
        for v in np.unique(cat[tr]):
            m = cat[tr] == v
            rate = (y[tr][m].sum() + alpha * y[tr].mean()) / (m.sum() + alpha)
            enc[va[cat[va] == v]] = rate
        enc[va] = np.where(np.isnan(enc[va]), y[tr].mean(), enc[va])  # unseen -> prior
    return enc

binc = oof_encode(cat, y, alpha)

# Per-category target rate (the dense signal):
rates = np.array([(y[cat == v].mean()) for v in range(5)])
print('per-bin target rate:', np.round(rates, 2))   # -> ~[0.95 0.86 0.71 0.33 0.08]

# Baseline: logistic regression on the raw label id vs on the bin-counted column.
acc_label = cross_val_score(LogisticRegression(max_iter=1000),
                            cat.reshape(-1, 1), y, cv=5).mean()
acc_binc  = cross_val_score(LogisticRegression(max_iter=1000),
                            binc.reshape(-1, 1), y, cv=5).mean()
print('label-encoded acc :', round(acc_label, 2))   # -> ~0.63
print('bin-counted acc   :', round(acc_binc, 2))    # -> ~0.90

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Your CTR model gets 0.95 AUC in cross-validation but 0.62 AUC in production after you add a device_ctr bin-counting feature. What is the most likely cause, and how do you fix it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Suspect target leakage: the encoding was probably computed using each row's own label. — _If $p_v$ for a row was built from a set that includes that row, the feature partly encodes the answer, inflating CV._
- Recompute the encoding out-of-fold (or leave-one-out, or from an earlier time window). — _Each row's $device\_ctr$ must come from OTHER rows only, so no label leaks into its own prediction._
- For click models, prefer the time-split: learn rates from a back-fill window, apply forward. — _It matches how production actually works (you only ever know the past), eliminating leakage realistically._

**Answer:** Target leakage. The bin-counted rate was computed using each row's own label, so the feature secretly carried the target &mdash; brilliant in CV, useless live. Fix: compute the encoding out-of-fold / leave-one-out, or from a separate earlier time window ("back-fill"). For CTR models the time-split is the most honest because production only ever knows the past.

</details>

**Problem 2.** A device appears twice in training and both impressions were clicked, giving a raw rate of 1.0. The global click rate is $p_0=0.05$. With smoothing $\alpha=10$, what encoded value should this device get, and why is the raw $1.0$ dangerous?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note $N_1=2$, $N_0=0$, $N_v=2$ — a rare category, so the raw rate is noise. — _Two clicks out of two is a fluke; a literal $1.0$ would tell the model this device always clicks._
- Apply additive smoothing: $\hat p_v=\frac{N_1+\alpha p_0}{N_v+\alpha}=\frac{2+10\cdot 0.05}{2+10}$. — _Adding $\alpha=10$ pseudo-rows of the prior drags the tiny-count estimate toward the global rate._
- Compute: $\frac{2+0.5}{12}=\frac{2.5}{12}\approx 0.208$. — _Above average (it did click) but nowhere near $1.0$ — the confidence is appropriately low._

**Answer:** Smoothed encoding $\hat p_v=\frac{2+10\cdot 0.05}{2+10}=\frac{2.5}{12}\approx \mathbf{0.21}$. The raw $1.0$ is dangerous because two-of-two is a rare-category fluke; trusting it overfits. Additive (Laplace) smoothing backs off to the prior $p_0$, shrinking low-count estimates toward the global rate so a $1/1$ can't pose as a guaranteed click.

</details>

**Problem 3.** At serving time a request arrives with a device_id that never appeared in training. What should the bin-counting feature return, and what must you make sure your pipeline does NOT do?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recognize this is the unseen-category case: $N_v=0$, no history for this value. — _There is no rate to compute, but the model still needs a number for this column._
- Return the global rate $p_0=P(y=1)$ as the fallback. — _It is the $N_v=0$ limit of the smoothing formula — "assume average until we learn otherwise."_
- Make sure the pipeline does not emit NaN or crash, and does not silently impute 0. — _NaN breaks the model; a hard 0 falsely claims this device never clicks, which is also wrong._

**Answer:** Return the global rate $p_0$ &mdash; the $N_v=0$ limit of $\hat p_v=\frac{N_1+\alpha p_0}{N_v+\alpha}$. The pipeline must not emit a NaN (it crashes the model) and must not silently fill 0 (that falsely claims the device never clicks). Falling back to the prior is the safe, calibrated default for any unseen value.

</details>